# Chapter 18: Privacy, Law, and Information Governance

> "Arguing that you don't care about the right to privacy because you have nothing to hide is no
> different than saying you don't care about free speech because you have nothing to say." Edward Snowden

---

## Learning Objectives

After completing this chapter, you will be able to:

1. State the eight Fair Information Practice Principles (FIPPs) and explain their historical origin.
2. Map FIPPs onto modern frameworks including GDPR, CCPA, and HIPAA.
3. Distinguish privacy from security and explain why security is a precondition for privacy.
4. Explain e-discovery preservation duties and the significance of the Zubulake decisions.
5. Apply a data-retention schedule and reason about lawful access to data.

## Key Terms

- **FIPPs**: Fair Information Practice Principles, the foundational privacy principles.
- **PII**: Personally Identifiable Information.
- **GDPR**: General Data Protection Regulation (EU Regulation 2016/679).
- **CCPA**: California Consumer Privacy Act.
- **HIPAA**: Health Insurance Portability and Accountability Act (United States).
- **DPIA**: Data Protection Impact Assessment.
- **DPO**: Data Protection Officer.
- **e-discovery**: Electronic discovery, the identification and production of electronically stored information in litigation.
- **RoPA**: Records of Processing Activities.

---

## 18.1 Privacy Is Not Security

Security protects data from unauthorized access, modification, and destruction. Privacy concerns whether
the collection and use of personal data is appropriate and authorized in the first place. The two are
related but distinct: an organization can hold perfectly secured data that it had no right to collect, and
strong security is a precondition for privacy because data that cannot be protected cannot be kept private.
This chapter addresses the privacy, legal, and governance dimensions that the technical chapters do not.

## 18.2 The Fair Information Practice Principles

The Fair Information Practice Principles were first articulated in the 1973 report of the U.S. Department
of Health, Education, and Welfare, "Records, Computers and the Rights of Citizens" {cite}`hew_fipps_1973`.
They were adopted and extended by the OECD in 1980 {cite}`oecd_privacy_1980` and now form the conceptual
backbone of essentially every modern privacy regime. The eight principles are:

1. **Collection Limitation**: collect only the minimum personal data needed for a clearly defined,
   legitimate purpose. GDPR calls this data minimization. A parking app does not need a full date of birth
   to take a payment.
2. **Data Quality**: personal data must be accurate, complete, and current relative to its use. Inaccurate
   records cause real harm such as wrong credit scores or erroneous criminal-history flags.
3. **Purpose Specification**: the purpose for collection must be stated at or before the time of collection
   and be specific enough that individuals understand the actual use. Vague purposes like "to improve our
   services" have been challenged as insufficient.
4. **Use Limitation**: data collected for one purpose may not be repurposed without fresh consent or legal
   authorization. This is the secondary-use prohibition central to HIPAA and GDPR.
5. **Security Safeguards**: data must be protected with technical and organizational measures scaled to its
   sensitivity. This is where the rest of this book connects to privacy.
6. **Openness and Transparency**: organizations must be open about their data practices and not collect or
   use data secretly. Notices must be accessible and written in plain language.
7. **Individual Participation**: individuals can confirm what is held about them, obtain a copy, challenge
   inaccuracies, and have data corrected or deleted. GDPR codifies these as the rights of access,
   rectification, and erasure (the right to be forgotten).
8. **Accountability**: organizations must demonstrate compliance, not merely assert it, through tools such
   as Data Protection Impact Assessments, Records of Processing Activities, and a Data Protection Officer.

## 18.3 FIPPs Across Modern Frameworks

The same principles surface under different names. Collection limitation appears as GDPR Article 5(1)(c)
data minimization, as the CCPA right to know what is collected, and as the HIPAA minimum-necessary
standard {cite}`eu_gdpr_2016`. Individual participation appears as GDPR data-subject rights in Articles
15 through 20, as the CCPA rights of access and deletion, and as the HIPAA right of access. Recognizing
that frameworks are variations on a shared core lets a practitioner reason about a new regulation by
asking which principle each clause implements.

## 18.4 Lawful Access and e-Discovery

When data becomes evidence, preservation duties attach. The Zubulake v. UBS Warburg decisions of 2003 to
2004 established foundational e-discovery obligations in the United States. The court found that the
defendant failed to preserve relevant emails and imposed severe sanctions, including an adverse-inference
instruction that allowed the jury to assume the missing emails were unfavorable; the plaintiff was
ultimately awarded a substantial verdict. The case drove the 2006 amendments to the Federal Rules of Civil
Procedure that created an explicit framework for electronically stored information. The practical lesson
for technical staff is that a litigation hold suspends ordinary data-destruction routines, and silent
deletion under an automated retention policy can itself become the most damaging fact in a case.

## 18.5 Why This Matters

Engineers build the systems that collect and process personal data, so privacy-by-design decisions are made
at the keyboard long before a lawyer ever sees them. A developer who minimizes collection, sets a retention
clock, and logs access is implementing FIPPs whether or not anyone uses that vocabulary. Understanding the
principles lets technical staff anticipate legal requirements rather than retrofit them after an incident.

## 18.6 News in Focus

Large regulatory penalties under GDPR have repeatedly turned on principles rather than on breaches alone.
Several of the headline fines issued since 2018 concerned a lawful-basis or transparency failure, that is,
processing personal data without an adequate legal basis or without clearly informing data subjects, rather
than a technical compromise. The pattern reinforces that privacy compliance is about the legitimacy of data
handling, not only about preventing intrusions.

## 18.7 Worked Example: A Retention Schedule

A retention schedule encodes the collection-limitation and use-limitation principles by deciding how long
each data category may be kept. The code below evaluates records against a simple schedule and flags those
that are overdue for deletion.


In [1]:
from dataclasses import dataclass
from datetime import date

# Retention schedule in days, derived from purpose and legal duty
RETENTION_DAYS = {
    "marketing_contact": 365,      # delete one year after last contact
    "payment_record":    365 * 7,  # financial records, seven years
    "access_log":        90,       # security logs, ninety days
    "job_applicant":     365 * 2,  # applicant data, two years
}

@dataclass
class Record:
    category: str
    last_activity: date

def days_held(rec, today):
    return (today - rec.last_activity).days

def disposition(rec, today):
    limit = RETENTION_DAYS.get(rec.category)
    if limit is None:
        return "UNCLASSIFIED: assign a retention category"
    age = days_held(rec, today)
    if age > limit:
        return f"DELETE: held {age} days, limit {limit}"
    return f"RETAIN: held {age} days, limit {limit}"

today = date(2026, 6, 1)
records = [
    Record("marketing_contact", date(2024, 1, 10)),
    Record("payment_record",    date(2023, 5, 1)),
    Record("access_log",        date(2026, 5, 1)),
    Record("job_applicant",     date(2023, 2, 1)),
]

for r in records:
    print(f"{r.category:18} {disposition(r, today)}")


marketing_contact  DELETE: held 873 days, limit 365
payment_record     RETAIN: held 1127 days, limit 2555
access_log         RETAIN: held 31 days, limit 90
job_applicant      DELETE: held 1216 days, limit 730


## 18.8 Review Questions (MCQ)

**Q1.** Which FIPP is most directly implemented by collecting only the data you actually need?
A. Accountability  B. Collection Limitation  C. Openness  D. Data Quality

**Q2.** The GDPR "right to be forgotten" corresponds to which FIPP?
A. Purpose Specification  B. Security Safeguards  C. Individual Participation  D. Use Limitation

**Q3.** A litigation hold primarily requires an organization to:
A. Delete old data faster  B. Suspend routine destruction of relevant data  C. Encrypt all data  D. Notify regulators

*Answers: Q1 B, Q2 C, Q3 B.*

## 18.9 Lab Assignment

Choose a small fictional service (for example a fitness app). List every category of personal data it would
collect, assign each a purpose and a retention period, and identify which FIPP justifies each choice. Then
write a one-paragraph plain-language privacy notice that would satisfy the openness principle.

## References

```{bibliography}
:filter: docname in docnames
```
